# 07 — RAG normalization
**Project:** Clinical Medication Extraction | **Phase 5b of the roadmap**

## Why this component exists — and why it's not decoration

Three unsolved problems from earlier notebooks, all the same shape:

| Problem | Where it appeared |
|---|---|
| `Ramelteon` detected but not normalizable | 05 — outside the lexicon |
| `Toprol XL` → `toprol`, losing the formulation | 03 — brand-name ambiguity |
| `vitamin` extracted from `vitamin D` | 03 — vague lexicon entries |

All three are **normalization** failures: mapping a surface string to a canonical concept. Your hand-built dictionary has ~200 entries. RxNorm has over 100,000, with brand↔generic relationships, ingredients, and dose forms.

**You cannot hand-write 100,000 mappings. You retrieve them.** That's the honest reason RAG belongs here — not because retrieval is fashionable, but because exact lookup fails on strings the dictionary has never seen, and semantic similarity degrades gracefully where exact match falls off a cliff.

This distinction matters in interviews. Most RAG portfolio projects are "chat with your PDF," where retrieval is bolted on for the buzzword. Here it's load-bearing: remove it and three named, measured failures come back.

## What retrieval buys over exact matching

Exact lookup is binary — `ramelteon` is either in your dict or it isn't. Embedding-based retrieval returns *ranked candidates by similarity*, so `Toprol XL` still ranks `metoprolol` first even though the strings differ. You then set a similarity threshold and decide what to do below it.

## Setup

In [ ]:
%pip install -q sentence-transformers chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently t

In [ ]:
import pandas as pd
import numpy as np
import re, json
from collections import Counter

IN_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    pass

BASE = '/content/drive/MyDrive/Clinical_notes/' if IN_COLAB else ''
WORK, SRC, GOLD, REF = BASE+'working/', BASE+'src/', BASE+'gold/', BASE+'reference/'

import os, sys
os.makedirs(REF, exist_ok=True)
sys.path.insert(0, SRC)
from rules_extractor import BRAND2GENERIC, GENERICS

print('starting lexicon size:', len(BRAND2GENERIC) + len(GENERICS))

ValueError: mount failed

---
# Part 1 — The reference vocabulary

## Getting RxNorm (do this properly)

RxNorm is free from the US National Library of Medicine. The **RxNorm Current Prescribable Content** release needs no UMLS licence and is the right subset for this task.

1. Go to the NLM RxNorm files page and download the Current Prescribable Content zip
2. Unzip; the file you want is `RXNCONSO.RRF` (pipe-delimited, no header)
3. Upload it to `reference/` in your Drive folder

The cell below uses RxNorm if present and falls back to your existing lexicon otherwise, so the notebook runs either way.

In [ ]:
RXN_PATH = REF + 'RXNCONSO.RRF'

RXNCONSO_COLS = ['RXCUI','LAT','TS','LUI','STT','SUI','ISPREF','RXAUI','SAUI','SCUI',
                 'SDUI','SAB','TTY','CODE','STR','SRL','SUPPRESS','CVF','EXTRA']

if os.path.exists(RXN_PATH):
    rxn = pd.read_csv(RXN_PATH, sep='|', header=None, names=RXNCONSO_COLS,
                      dtype=str, index_col=False)
    rxn = rxn[(rxn['SAB'] == 'RXNORM') & (rxn['SUPPRESS'] != 'Y')]
    # TTY: IN=ingredient, BN=brand name, PIN=precise ingredient, SCD/SBD=clinical/branded drug
    keep = rxn[rxn['TTY'].isin(['IN','PIN','BN','MIN'])][['RXCUI','TTY','STR']].dropna()
    concepts = (keep.assign(name=lambda d: d['STR'].str.strip())
                    .drop_duplicates('name')[['RXCUI','TTY','name']]
                    .reset_index(drop=True))
    print(f'RxNorm concepts loaded: {len(concepts):,}')
else:
    print('RXNCONSO.RRF not found — falling back to the notebook-03 lexicon.')
    rows = ([{'RXCUI': f'L{i}', 'TTY':'BN', 'name': b} for i, b in enumerate(BRAND2GENERIC)]
            + [{'RXCUI': f'L{i+1000}', 'TTY':'IN', 'name': g} for i, g in enumerate(sorted(GENERICS))])
    concepts = pd.DataFrame(rows)
    print(f'Fallback vocabulary: {len(concepts)} concepts')

concepts.head()

---
# Part 2 — Embeddings

## Choosing a model

`all-MiniLM-L6-v2` — small, fast, and good at short-string similarity. **Not** a clinical model, and that's a deliberate choice worth defending: we're matching drug *name strings*, not clinical meaning. Character and morphological similarity is what matters (`Toprol XL` ↔ `metoprolol` via `toprol`), and a general sentence model handles that well while being 10× faster than a biomedical encoder.

**If your evaluation below shows poor recall@5, this is the first thing to swap** — try a biomedical embedding model and re-measure. Having the harness in place is what makes that a 10-minute experiment instead of a guess.

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer('all-MiniLM-L6-v2')
names = concepts['name'].tolist()

emb = embedder.encode(names, batch_size=256, show_progress_bar=True,
                      normalize_embeddings=True)      # normalized -> dot product = cosine
print('embedding matrix:', emb.shape)

In [ ]:
def retrieve(query, k=5):
    """Return top-k concepts by cosine similarity."""
    q = embedder.encode([query], normalize_embeddings=True)[0]
    sims = emb @ q                                     # cosine, since both normalized
    idx = np.argpartition(-sims, min(k, len(sims)-1))[:k]
    idx = idx[np.argsort(-sims[idx])]
    return [{'rxcui': concepts.iloc[i]['RXCUI'], 'name': concepts.iloc[i]['name'],
             'tty': concepts.iloc[i]['TTY'], 'score': float(sims[i])} for i in idx]

for q in ['Toprol XL', 'Ramelteon', 'vitamin D', 'lasix', 'ASA']:
    hits = retrieve(q, k=3)
    print(f'{q:12} -> ' + ', '.join(f"{h['name']}({h['score']:.2f})" for h in hits))

**Why `normalize_embeddings=True` and a dot product.** Cosine similarity is the dot product of unit vectors. Normalizing once at encode time turns every later similarity computation into a single matrix multiply — much faster than computing norms per query, and the reason this scales to 100k concepts without a vector database at this size.

**When you'd actually need Chroma or FAISS:** above roughly 10⁶ vectors, or when you need persistence and filtering. At RxNorm scale, a NumPy matrix is genuinely the right tool, and saying so in an interview signals judgment rather than tool-collecting.

---
# Part 3 — Evaluate retrieval *separately*

## The methodological point that matters most here

It is tempting to measure only end-to-end accuracy. **Don't.** If end-to-end F1 drops, you can't tell whether retrieval returned the wrong candidates or the downstream logic mishandled the right ones.

Measuring retrieval in isolation with **recall@k** and **MRR** localizes the failure:
- **recall@k** — is the correct concept anywhere in the top k? If recall@5 is low, the embedding model or the vocabulary is wrong.
- **MRR** (mean reciprocal rank) — how high does it rank? If recall@5 is high but recall@1 is low, retrieval works and your *selection* rule needs work.

Separating retrieval metrics from end-to-end metrics is precisely what job postings mean by "RAG evaluation," and it's the part most portfolio projects skip.

In [ ]:
def recall_at_k(retrieved, truth, ks=(1, 3, 5)):
    out = {}
    for k in ks:
        hits = sum(1 for cands, t in zip(retrieved, truth) if t in cands[:k])
        out[f'recall@{k}'] = round(hits / len(truth), 3) if truth else None
    return out

def mrr(retrieved, truth):
    total = sum(1.0 / (c.index(t) + 1) for c, t in zip(retrieved, truth) if t in c)
    return round(total / len(truth), 3) if truth else None

# sanity check on a hand-checkable case
_r = [['atorvastatin','pravastatin','simvastatin'], ['metoprolol','metolazone','methadone'],
      ['furosemide','fluoxetine','fluconazole'], ['docusate','dobutamine','donepezil']]
_t = ['atorvastatin', 'metolazone', 'furosemide', 'digoxin']
print(recall_at_k(_r, _t), 'MRR =', mrr(_r, _t))
print('expected: recall@1=0.5 (2 of 4 rank first), recall@3=0.75, MRR=0.625')

## Build the mapping evaluation set

50 surface forms with their correct canonical names, hand-labelled. This takes about 30 minutes and it's the only way to know whether retrieval works.

**Sample the surface forms from your actual extractor output, not from a textbook list.** You want the strings your system really produces — including the awkward ones.

In [ ]:
ex_ner_path = WORK + 'extractions_ner.parquet'
pool = []
for p in [WORK + 'extractions_rules.parquet', ex_ner_path, WORK + 'extractions_llm.parquet']:
    if os.path.exists(p):
        pool.append(pd.read_parquet(p)[['drug_text']])
surface_forms = (pd.concat(pool)['drug_text'].dropna().str.strip()
                 if pool else pd.Series(dtype=str))
top_surfaces = surface_forms.value_counts().head(80)

MAP_FILE = GOLD + 'mapping_eval.csv'
if not os.path.exists(MAP_FILE):
    tmpl = pd.DataFrame({'surface': top_surfaces.index[:50],
                         'n_occurrences': top_surfaces.values[:50],
                         'correct_name': ''})
    tmpl.to_csv(MAP_FILE, index=False)
    print(f'Wrote {MAP_FILE} — fill the correct_name column (~30 min), then re-run.')
    print(tmpl.head(10).to_string(index=False))
else:
    print(f'{MAP_FILE} exists.')

In [ ]:
if os.path.exists(MAP_FILE):
    mp = pd.read_csv(MAP_FILE)
    mp = mp[mp['correct_name'].notna() & (mp['correct_name'].astype(str).str.strip() != '')]
    if len(mp):
        retrieved = [[h['name'].lower() for h in retrieve(s, k=5)] for s in mp['surface']]
        truth = mp['correct_name'].str.lower().str.strip().tolist()
        print('RETRIEVAL QUALITY  (n =', len(mp), ')')
        print(' ', recall_at_k(retrieved, truth))
        print('  MRR =', mrr(retrieved, truth))
        print()
        misses = [(s, t, r[:3]) for s, t, r in zip(mp['surface'], truth, retrieved) if t not in r]
        print(f'{len(misses)} surface forms where the correct concept is not in the top 5:')
        for s, t, r in misses[:10]:
            print(f"  {s:22} want={t:22} got={r}")
    else:
        print('mapping_eval.csv is empty — fill correct_name first.')

### How to act on these numbers

- **recall@5 below ~0.85** → the retrieval side is the problem. Try a biomedical embedding model, or check whether the missing concepts exist in your vocabulary at all (a vocabulary gap looks identical to a retrieval failure in this metric — check before you swap models).
- **recall@5 high, recall@1 low** → retrieval is fine; your *selection* rule needs work. That's the threshold below.
- **The miss list is the more useful output than the aggregate.** Patterns in it — abbreviations, combination products, dose forms — each imply a different fix.

---
# Part 4 — From retrieval to a decision

Retrieval gives ranked candidates. Something must decide: accept the top hit, or abstain?

**A similarity threshold, with explicit abstention.** Below the threshold we return `None` and flag `needs_review` rather than forcing a wrong mapping — the same principle as `attrs_ambiguous` in notebook 03. **In clinical systems, "I don't know" is a valid and often correct output**, and a normalization system that silently maps `Ramelteon` to the nearest-looking statin is far more dangerous than one that abstains.

In [ ]:
SIM_THRESHOLD = 0.62          # tune on the sweep below

def normalize_via_rag(surface, threshold=SIM_THRESHOLD):
    hits = retrieve(surface, k=5)
    if not hits or hits[0]['score'] < threshold:
        return {'normalized': None, 'rxcui': None, 'score': hits[0]['score'] if hits else 0.0,
                'needs_review': True, 'candidates': [h['name'] for h in hits[:3]]}
    top = hits[0]
    return {'normalized': top['name'].lower(), 'rxcui': top['rxcui'], 'score': top['score'],
            'needs_review': False, 'candidates': [h['name'] for h in hits[:3]]}

for q in ['Toprol XL', 'Ramelteon', 'vitamin D', 'Lasix', 'qwertyzol']:
    r = normalize_via_rag(q)
    print(f"{q:12} -> {str(r['normalized']):20} score={r['score']:.2f} "
          f"review={r['needs_review']}")

In [ ]:
# Threshold sweep on the labelled mapping set
if os.path.exists(MAP_FILE):
    mp = pd.read_csv(MAP_FILE)
    mp = mp[mp['correct_name'].notna() & (mp['correct_name'].astype(str).str.strip() != '')]
    if len(mp):
        rows = []
        for thr in [0.50, 0.55, 0.60, 0.65, 0.70, 0.75]:
            acc = abstain = 0
            for s, t in zip(mp['surface'], mp['correct_name'].str.lower().str.strip()):
                r = normalize_via_rag(s, threshold=thr)
                if r['needs_review']:
                    abstain += 1
                elif r['normalized'] == t:
                    acc += 1
            answered = len(mp) - abstain
            rows.append({'threshold': thr, 'answered': answered,
                         'abstained': abstain,
                         'accuracy_when_answered': round(acc / answered, 3) if answered else None,
                         'coverage': round(answered / len(mp), 3)})
        print(pd.DataFrame(rows).to_string(index=False))
        print()
        print('This is a coverage/accuracy tradeoff, not an accuracy curve.')
        print('Pick by consequence: a wrong normalization is worse than a flagged one.')

### Coverage vs. accuracy is the real decision

Raising the threshold makes the system answer less often but be right more often when it does. Lowering it does the reverse. **There is no "best" point in the abstract** — it depends on what happens to an abstention downstream.

- If `needs_review` items go to a human queue → favour accuracy, abstain freely.
- If abstentions silently drop the medication → favour coverage, because a dropped drug is worse than a flagged guess.

**Designing the downstream handling of abstentions is part of choosing the threshold.** Reporting the threshold without saying what happens to the abstained cases is an incomplete answer.

---
# Part 5 — Apply and save

In [ ]:
if os.path.exists(ex_ner_path):
    ex = pd.read_parquet(ex_ner_path)
    unresolved = ex[~ex['in_lexicon']].copy() if 'in_lexicon' in ex.columns else ex.copy()
    print(f'Normalizing {len(unresolved)} previously-unresolved extractions...')

    uniq = unresolved['drug_text'].dropna().unique()
    cache = {s: normalize_via_rag(s) for s in uniq}        # cache: same string, same answer

    unresolved['rag_normalized'] = unresolved['drug_text'].map(lambda s: cache.get(s, {}).get('normalized'))
    unresolved['rag_score'] = unresolved['drug_text'].map(lambda s: cache.get(s, {}).get('score'))
    unresolved['needs_review'] = unresolved['drug_text'].map(lambda s: cache.get(s, {}).get('needs_review'))

    resolved = unresolved['rag_normalized'].notna().sum()
    print(f'  resolved: {resolved} / {len(unresolved)} ({resolved/max(len(unresolved),1):.1%})')
    print(f'  flagged for review: {unresolved["needs_review"].sum()}')
    print()
    print('Newly resolved examples:')
    print(unresolved[unresolved['rag_normalized'].notna()]
          [['drug_text','rag_normalized','rag_score']].drop_duplicates('drug_text').head(12).to_string(index=False))
    unresolved.to_parquet(WORK + 'extractions_rag_normalized.parquet')
else:
    print('Run notebook 05 first to produce extractions_ner.parquet.')

In [ ]:
rag_src = '''"""RAG normalization over a drug vocabulary."""
import numpy as np


class DrugNormalizer:
    def __init__(self, concepts, embedder, threshold=0.62):
        self.concepts = concepts.reset_index(drop=True)
        self.embedder = embedder
        self.threshold = threshold
        self.emb = embedder.encode(self.concepts["name"].tolist(),
                                   batch_size=256, normalize_embeddings=True)
        self._cache = {}

    def retrieve(self, query, k=5):
        q = self.embedder.encode([query], normalize_embeddings=True)[0]
        sims = self.emb @ q
        idx = np.argpartition(-sims, min(k, len(sims) - 1))[:k]
        idx = idx[np.argsort(-sims[idx])]
        return [{"rxcui": self.concepts.iloc[i]["RXCUI"],
                 "name": self.concepts.iloc[i]["name"],
                 "score": float(sims[i])} for i in idx]

    def normalize(self, surface, threshold=None):
        if surface in self._cache:
            return self._cache[surface]
        thr = self.threshold if threshold is None else threshold
        hits = self.retrieve(surface, k=5)
        if not hits or hits[0]["score"] < thr:
            out = {"normalized": None, "rxcui": None,
                   "score": hits[0]["score"] if hits else 0.0,
                   "needs_review": True,
                   "candidates": [h["name"] for h in hits[:3]]}
        else:
            top = hits[0]
            out = {"normalized": top["name"].lower(), "rxcui": top["rxcui"],
                   "score": top["score"], "needs_review": False,
                   "candidates": [h["name"] for h in hits[:3]]}
        self._cache[surface] = out
        return out


def recall_at_k(retrieved, truth, ks=(1, 3, 5)):
    return {f"recall@{k}": round(sum(1 for c, t in zip(retrieved, truth) if t in c[:k]) / len(truth), 3)
            for k in ks} if truth else {}


def mrr(retrieved, truth):
    if not truth:
        return None
    total = sum(1.0 / (c.index(t) + 1) for c, t in zip(retrieved, truth) if t in c)
    return round(total / len(truth), 3)
'''

with open(SRC + 'rag_normalizer.py', 'w') as f:
    f.write(rag_src)
print('Wrote src/rag_normalizer.py')

---
## What you built

An embedding-based normalizer over a drug vocabulary, with retrieval evaluated in isolation (recall@k, MRR), a threshold sweep framed as coverage vs. accuracy, and explicit abstention on low-confidence matches.

**The four transferable ideas:**
1. **RAG earns its place by solving a named failure, not by being fashionable.** Three documented problems from notebooks 03 and 05 are what justify it.
2. **Evaluate retrieval separately from end-to-end.** Otherwise you cannot tell which half is broken.
3. **A NumPy matrix is a vector database at this scale.** Reach for Chroma/FAISS when the numbers demand it, not by default.
4. **Abstention is a feature.** `needs_review` beats a confident wrong mapping, and how abstentions are handled downstream is part of choosing the threshold.

**For `decisions.md`:**
- RxNorm Current Prescribable Content as vocabulary (no UMLS licence needed); fallback to the curated lexicon
- `all-MiniLM-L6-v2` chosen for string similarity, not clinical semantics; swap to a biomedical encoder if recall@5 < 0.85
- Retrieval evaluated separately (recall@k, MRR) on a 50-item hand-labelled mapping set
- Similarity threshold set by coverage/accuracy tradeoff, not max accuracy; low-confidence matches abstain with `needs_review`
- No vector DB — normalized embeddings + matrix multiply is sufficient at ~10⁵ concepts

**Next: `08_hybrid_and_final.ipynb`** — combine everything, score the hybrid, and write the ADR.